# 03 — Graphs, shortest paths and network skimming

Every path-based computation in AequilibraE runs on a **Graph** — a compiled,
Cython-backed representation of the network for one mode. In this notebook we:

1. build graphs from the Sioux Falls project;
2. compute a shortest path between two nodes and map it;
3. **skim** the network: compute zone-to-zone cost matrices (time, distance);
4. store the skims in the project.

Skim matrices are the backbone of demand modeling: trip distribution (notebook 04)
consumes them as impedance.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from aequilibrae.utils.create_example import create_example

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "sioux_falls")

project.network.build_graphs()
graph = project.network.graphs["c"]        # 'c' = car

# Minimise free-flow time; skim both time and distance along the way
graph.set_graph("free_flow_time")
graph.set_skimming(["free_flow_time", "distance"])

# Sioux Falls quirk: all nodes are centroids, so allow paths through centroids
graph.set_blocked_centroid_flows(False)
graph

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are settin

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are settin

## A single shortest path

`PathResults` computes one origin's tree and lets us extract the path to any destination.


In [2]:
from aequilibrae.paths import PathResults

res = PathResults()
res.prepare(graph)
res.compute_path(1, 17)     # from node 1 to node 17

print("nodes :", res.path_nodes)
print("links :", res.path)
print(f"cost  : {res.milepost[-1]:.2f} minutes")

nodes : [ 1  2  6  8 16 17]
links : [ 1  4 16 22 49]
cost  : 20.00 minutes


In [3]:
# Maps, cartographic standards and UK geography helpers.
# Model logic stays in the notebook; everything reusable lives in notebooks/uktools/.
from uktools import *


In [4]:
# field()/constant() symbology builders come from the map helper cell

links = project.network.links.data
path_links = links[links.link_id.isin(res.path)]

doc = new_map(links, zoom=12)
add_gdf(doc, links, "network", opacity=0.5, symbology=[[constant("#94a3b8").encoding("stroke")]])
add_gdf(doc, path_links, "shortest path", symbology=[[constant("#dc2626").encoding("stroke")]])
doc

[interactive offline map - run the notebook to display]

## Skimming the whole network

`NetworkSkimming` runs one shortest-path tree per origin (in parallel) and collects
the skimmed fields into a zone-by-zone matrix.


In [5]:
from aequilibrae.paths import NetworkSkimming

skm = NetworkSkimming(graph)
skm.execute()

skims = skm.results.skims
print(skims.names)          # one matrix core per skimmed field
skims.get_matrix("free_flow_time")[:5, :5]

[interactive offline map - run the notebook to display]

['free_flow_time', 'distance']


array([[ 0.,  6.,  4.,  8., 10.],
       [ 6.,  0., 10., 11.,  9.],
       [ 4., 10.,  0.,  4.,  6.],
       [ 8., 11.,  4.,  0.,  2.],
       [10.,  9.,  6.,  2.,  0.]])

In [6]:
# Persist into the project so later notebooks (and colleagues) can reuse them
skm.save_to_project("base_skims")
project.matrices.list()[["name", "file_name", "cores"]]

,name,file_name,cores
0,demand_omx,demand.omx,1
1,demand_mc,demand_mc.omx,3
2,skims,skims.omx,2
3,demand_aem,demand.aem,1
4,base_skims,base_skims.omx,2


In [7]:
project.close()

---
**Next:** [04 — Trip distribution](04_trip_distribution.ipynb): turning trip totals into
a full origin-destination matrix with gravity models and IPF.
